# Results Analysis

This notebook analyzes the results from the trained models and provides detailed performance metrics.

In [ ]:
# Import necessary libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

print("Libraries imported successfully.")

## 1. Load Trained Models and Test Data

In [ ]:
# Load test data
test_df = pd.read_csv('../data/processed/test.csv')
X_test = test_df.drop('label', axis=1).values
y_test = test_df['label'].values

print(f"Test data loaded: {X_test.shape}")

# Load preprocessor
preprocessor = joblib.load('../models_saved/preprocessor.pkl')

# Load all models
models_dir = Path('../models_saved')
models = {}
for model_file in models_dir.glob('*.pkl'):
    if model_file.name != 'preprocessor.pkl':
        model_name = model_file.stem
        models[model_name] = joblib.load(str(model_file))
        print(f"Loaded: {model_name}")

## 2. Evaluate All Models

In [ ]:
# Evaluate each model
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # ROC AUC if available
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = None
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'ROC AUC': roc_auc
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1 Score', ascending=False)
print("\nModel Performance Summary:")
print(results_df.to_string(index=False))

## 3. Visualize Performance Metrics

In [ ]:
# Plot performance comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    sns.barplot(
        data=results_df,
        x=metric,
        y='Model',
        palette='viridis',
        ax=ax
    )
    ax.set_title(f'{metric} by Model', fontweight='bold')
    ax.set_xlabel(metric)
    ax.set_xlim(0.8, 1.0)
    ax.spines[['top', 'right']].set_visible(False)
    
    # Add value labels
    for i, v in enumerate(results_df[metric]):
        ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/performance_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: performance_metrics_comparison.png")

## 4. ROC Curves

In [ ]:
# Plot ROC curves for models with predict_proba
fig, ax = plt.subplots(figsize=(10, 8))

for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = roc_auc_score(y_test, y_proba)
        ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('ROC Curves - Model Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../reports/figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roc_curves.png")

## 5. Best Model Detailed Analysis

In [ ]:
# Get best model
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

print(f"\nBest Model: {best_model_name}")
print("="*60)

# Detailed classification report
y_pred = best_model.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing'], digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

## 6. Feature Importance (Tree-based Models)

In [ ]:
# Plot feature importance for tree-based models
tree_models = ['random_forest', 'xgboost', 'decision_tree']

for model_name in tree_models:
    if model_name in models:
        model = models[model_name]
        
        if hasattr(model, 'feature_importances_'):
            importances = pd.Series(model.feature_importances_, index=test_df.columns[:-1])
            importances = importances.nlargest(15).sort_values()
            
            fig, ax = plt.subplots(figsize=(10, 8))
            colors = sns.color_palette("YlOrRd", len(importances))
            importances.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
            ax.set_title(f'Top 15 Feature Importances - {model_name}', fontweight='bold')
            ax.set_xlabel('Importance')
            ax.spines[['top', 'right']].set_visible(False)
            ax.xaxis.grid(True, linestyle='--', alpha=0.7)
            ax.set_axisbelow(True)
            
            plt.tight_layout()
            plt.savefig(f'../reports/figures/feature_importance_{model_name}.png', dpi=150, bbox_inches='tight')
            plt.show()
            print(f"Saved: feature_importance_{model_name}.png")

## Summary

Results analysis complete! Key findings:

- Best performing model: {best_model_name}
- Test accuracy: {results_df.iloc[0]['Accuracy']:.4f}
- F1 Score: {results_df.iloc[0]['F1 Score']:.4f}

All visualizations saved to reports/figures/.